# 05 · Train the queue  (steps 10-12)
**Accelerator: GPU T4 x2.** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


Set `QUEUE` and run. Each session trains for ~11 h, saving a checkpoint every 30 min and at every epoch, and
uploads to HF in the background. A half-finished job continues exactly where it stopped. Two T4s train two
different jobs. When this account's GPU quota is used up, run the same notebook on another account (same
`HF_REPO`, its own `HF_TOKEN` secret): it picks up the queue automatically.

Queues: `queues/main.yaml` (main rows, seeds 0→1→2), `queues/ablations.yaml`, `queues/nyud.yaml`,
`queues/multicue.yaml`.

In [ ]:
# ==== EDIT ONLY THIS CELL (keep the same values on every Kaggle account) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # private Hugging Face dataset repo; created automatically
ACCOUNT = "account-1"                         # any label; recorded in progress.json (who trained what)
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Kaggle setup (once per account): Add-ons -> Secrets -> add HF_TOKEN (a WRITE token) and tick it for this
# notebook; Settings -> Internet ON; Settings -> Accelerator as noted in the title.

In [ ]:
QUEUE = "queues/main.yaml"
HOURS = 11.25            # whole-session budget, counted from the first cell (Kaggle limit: 12 h)

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=0.25", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
if not os.path.exists(REPO):
    attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
    if attached:                                   # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
        CODE_SOURCE = "kaggle dataset"
    elif REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
        CODE_SOURCE = "git"
    else:                                          # copy pushed to Hugging Face by notebook 01
        from huggingface_hub import snapshot_download
        snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                          token=os.environ["HF_TOKEN"])
        shutil.copytree(f"{W}/hf_code/code", REPO)
        CODE_SOURCE = "huggingface"
    print("code from:", CODE_SOURCE)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py

In [ ]:
!python -m pedger_kd.sync status

In [ ]:
!python -m pedger_kd.jobs --queue {QUEUE} --ws {WS} --hours {HOURS} --session-start {SESSION_START} --gpus auto --label {ACCOUNT} --squash

In [ ]:
!python -m pedger_kd.sync status

In [ ]:
# keep the notebook output small (everything important is already on Hugging Face)
for d in ["ws/data", "ws/teachers", "ws/.staging", "hf_code", "work", "BSR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h")